In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
import warnings

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 10 # Bumped to 10-fold for better stacking generalization

# ── Load data ──────────────────────────────────────────────────────────────────
train_df = pd.read_csv("train_assignment.csv")
test_df = pd.read_csv("test_assignment.csv")

# ── Feature engineering ────────────────────────────────────────────────────────
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Total academic volume
    df["cu_total_enrolled"] = df["cu1_enrolled"] + df["cu2_enrolled"]
    df["cu_total_evaluations"] = df["cu1_evaluations"] + df["cu2_evaluations"]
    df["cu_total_approved"] = df["cu1_approved"] + df["cu2_approved"]
    df["cu_total_credited"] = df["cu1_credited"] + df["cu2_credited"]
    
    # Precise smoothing for rates
    eps = 1e-6
    df["cu1_approval_rate"] = df["cu1_approved"] / (df["cu1_enrolled"] + eps)
    df["cu2_approval_rate"] = df["cu2_approved"] / (df["cu2_enrolled"] + eps)
    df["cu_total_approval_rate"] = df["cu_total_approved"] / (df["cu_total_enrolled"] + eps)

    # Grade dynamics
    df["cu_grade_mean"] = (df["cu1_grade"] + df["cu2_grade"]) / 2.0
    df["cu_grade_diff"] = df["cu2_grade"] - df["cu1_grade"]
    df["cu_approved_diff"] = df["cu2_approved"] - df["cu1_approved"]

    # Trajectory & Risk
    df["academic_trajectory"] = df["cu_grade_diff"] * df["cu_approved_diff"]
    df["financial_distress"] = (df["debtor_flag"] == 1) & (df["tuition_up_to_date_flag"] == 0)
    df["financial_distress"] = df["financial_distress"].astype(int)

    # Flags for completely inactive semesters
    df["zero_approved_sem1"] = (df["cu1_approved"] == 0).astype(int)
    df["zero_approved_sem2"] = (df["cu2_approved"] == 0).astype(int)
    
    return df

train_fe = add_features(train_df)
test_fe = add_features(test_df)

target_col = "Target"
id_col = "id"

# ── Categorical Formatting ─────────────────────────────────────────────────────
# Identify codes/flags to pass natively to CatBoost & LightGBM
cat_cols = [c for c in train_fe.columns if c.endswith('_code') or c.endswith('_flag')]

for c in cat_cols:
    train_fe[c] = train_fe[c].astype(int).astype('category')
    test_fe[c] = test_fe[c].astype(int).astype('category')

feature_cols = [c for c in train_fe.columns if c not in [target_col, id_col]]

X = train_fe[feature_cols]
X_test = test_fe[feature_cols]

le = LabelEncoder()
y = le.fit_transform(train_fe[target_col])
class_names = le.classes_
n_classes = len(class_names)

print(f"Features: {len(feature_cols)}, Train: {len(X)}, Test: {len(X_test)}")

# ── Model definitions ─────────────────────────────────────────────────────────
def get_models():
    # Extracted purely for categories as indices
    cat_indices = [X.columns.get_loc(c) for c in cat_cols]
    
    return {
        "xgb1": XGBClassifier(
            objective="multi:softprob", n_estimators=1500, learning_rate=0.03,
            max_depth=6, subsample=0.8, colsample_bytree=0.7, min_child_weight=4,
            random_state=RANDOM_STATE, enable_categorical=True, n_jobs=-1,
            early_stopping_rounds=100, tree_method="hist"
        ),
        "lgbm1": LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, max_depth=8, num_leaves=63,
            subsample=0.8, colsample_bytree=0.7, random_state=RANDOM_STATE, 
            n_jobs=-1, verbose=-1, categorical_feature=cat_indices
        ),
        "cat1": CatBoostClassifier(
            iterations=2000, learning_rate=0.04, depth=6, l2_leaf_reg=4.0,
            cat_features=cat_cols, random_seed=RANDOM_STATE, verbose=0,
            early_stopping_rounds=100, task_type="CPU"
        ),
        "hgb": HistGradientBoostingClassifier(
            learning_rate=0.03, max_iter=1500, max_depth=7, l2_regularization=2.0,
            categorical_features=cat_indices, random_state=RANDOM_STATE,
            early_stopping=True, n_iter_no_change=50
        )
    }

# ── K-fold stacking ───────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
model_names = list(get_models().keys())

oof_probas = {name: np.zeros((len(X), n_classes), dtype=np.float64) for name in model_names}
test_fold_probas = {name: [] for name in model_names}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    
    print(f"Fold {fold}/{N_SPLITS}...")
    fold_models = get_models()
    
    for name, model in fold_models.items():
        if "xgb" in name:
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        elif "lgbm" in name:
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], 
                      callbacks=[__import__("lightgbm").early_stopping(100, verbose=False)])
        elif "cat" in name:
            model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
        else:
            model.fit(X_tr, y_tr)

        oof_probas[name][va_idx] = model.predict_proba(X_va)
        test_fold_probas[name].append(model.predict_proba(X_test))

# ── Evaluation & Ensemble Generation ───────────────────────────────────────────
test_probas_mean = {name: np.mean(np.stack(test_fold_probas[name], axis=0), axis=0) for name in model_names}

stack_X = np.hstack([oof_probas[name] for name in model_names])
stack_X_test = np.hstack([test_probas_mean[name] for name in model_names])

# Log transform predictions to tame aggressive extremes before feeding to Stack
epsilon = 1e-10
stack_X_log = np.log(stack_X + epsilon)
stack_X_test_log = np.log(stack_X_test + epsilon)

meta_clf = LogisticRegression(C=0.5, max_iter=2000, solver="lbfgs")
meta_clf.fit(stack_X_log, y)

oof_stack_pred = meta_clf.predict(stack_X_log)
test_stack_prob = meta_clf.predict_proba(stack_X_test_log)

acc_stack = accuracy_score(y, oof_stack_pred)
f1_stack = f1_score(y, oof_stack_pred, average="macro")
print(f"\n✅ Log-Stacking Ensemble -> Accuracy: {acc_stack:.5f}, Macro-F1: {f1_stack:.5f}")

# ── Generate submission ──────────────────────────────────────────────────────
test_pred = le.inverse_transform(np.argmax(test_stack_prob, axis=1))
submission = pd.DataFrame({"id": test_df["id"], "Target": test_pred})
submission.to_csv("submission_tuned.csv", index=False)
print(f"Saved submission_tuned.csv")

Features: 50, Train: 76518, Test: 51012
Fold 1/10...


KeyboardInterrupt: 